# Fase 2 — Notebook 01: Exploração dos Dados

**Tech Challenge FIAP — VRP para Saúde da Mulher**

Este notebook explora os dados sintéticos gerados para o sistema de otimização de rotas:
- Distribuição geográfica dos pontos de atendimento em São Paulo
- Análise por tipo de atendimento e prioridade
- Distribuição de demanda e janelas de tempo
- Visualização da matriz de distâncias

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from fase2_vrp.src.data_generator import (
    generate_service_points, get_depot, build_distance_matrix,
    summarize_points, TYPE_LABELS, TYPE_COLORS_PLACEHOLDER
)

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print('Imports OK')

In [ ]:
# Gerar dados sintéticos
SEED = 42
N_POINTS = 20

depot  = get_depot()
points = generate_service_points(n_points=N_POINTS, seed=SEED)
dist_matrix = build_distance_matrix(depot, points)

summarize_points(points)
print(f'Depósito: {depot.name} | lat={depot.lat}, lon={depot.lon}')

In [ ]:
# Converter para DataFrame para análise
df = pd.DataFrame([
    {
        'id': p.id,
        'name': p.name,
        'type': p.type,
        'type_label': TYPE_LABELS[p.type],
        'priority': p.priority,
        'lat': p.lat,
        'lon': p.lon,
        'tw_start': p.time_window[0],
        'tw_end': p.time_window[1],
        'tw_duration': p.time_window[1] - p.time_window[0],
        'demand': p.demand,
        'service_time': p.service_time,
    }
    for p in points
])

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Estatísticas descritivas
print('\n=== Estatísticas Numéricas ===')
df[['priority', 'demand', 'service_time', 'tw_duration']].describe().round(2)

In [ ]:
# Distribuição por tipo de atendimento
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Contagem por tipo
type_counts = df['type_label'].value_counts()
colors_bar = ['#E63946', '#FF7F50', '#2196F3', '#4CAF50']
type_counts.plot(kind='bar', ax=axes[0], color=colors_bar[:len(type_counts)],
                 edgecolor='white')
axes[0].set_title('Pontos por Tipo de Atendimento')
axes[0].set_xlabel('')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=30)

# 2. Distribuição de demanda por tipo
df.boxplot(column='demand', by='type_label', ax=axes[1])
axes[1].set_title('Distribuição de Demanda por Tipo')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

# 3. Distribuição de janelas de tempo
for stype, color in zip(['emergencia_obstetrica', 'violencia_domestica',
                          'medicamento_hormonal', 'pos_parto'],
                         ['#E63946', '#FF7F50', '#2196F3', '#4CAF50']):
    subset = df[df['type'] == stype]
    if len(subset) > 0:
        axes[2].scatter(subset['tw_start'], subset['tw_end'],
                       label=TYPE_LABELS.get(stype, stype), color=color, s=80)
axes[2].set_title('Janelas de Tempo por Tipo')
axes[2].set_xlabel('Início da janela (h)')
axes[2].set_ylabel('Fim da janela (h)')
axes[2].legend(fontsize=8)

plt.suptitle('Análise dos Pontos de Atendimento — Saúde da Mulher', fontsize=13)
plt.tight_layout()
plt.savefig('../results/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em results/data_exploration.png')

In [ ]:
# Mapa de exploração (todos os pontos, sem rota)
import numpy as np

center_lat = np.mean([p.lat for p in points])
center_lon = np.mean([p.lon for p in points])

TYPE_COLORS = {
    'emergencia_obstetrica': '#E63946',
    'violencia_domestica':   '#FF7F50',
    'medicamento_hormonal':  '#2196F3',
    'pos_parto':             '#4CAF50',
}

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

# Depósito
folium.Marker(
    [depot.lat, depot.lon],
    popup=depot.name,
    icon=folium.Icon(color='purple', icon='home', prefix='fa'),
    tooltip='Depósito Central'
).add_to(m)

# Pontos de atendimento
for p in points:
    color = TYPE_COLORS[p.type]
    folium.CircleMarker(
        location=[p.lat, p.lon],
        radius=10,
        color=color, fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(
            f'<b>{p.name}</b><br>Tipo: {TYPE_LABELS[p.type]}<br>'
            f'Prioridade: {p.priority}<br>Demanda: {p.demand} un.',
            max_width=200
        ),
        tooltip=f'{p.name} (Prio {p.priority})'
    ).add_to(m)

m.save('../results/exploration_map.html')
print('Mapa salvo em results/exploration_map.html')
m

In [ ]:
# Heatmap da matriz de distâncias
import numpy as np

dist_array = np.array(dist_matrix)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    dist_array,
    ax=ax,
    cmap='YlOrRd',
    annot=False,
    fmt='.0f',
    cbar_kws={'label': 'Distância (km)'}
)
ax.set_title('Matriz de Distâncias entre Nós (Depósito=0, Pontos=1..N)', fontsize=12)
ax.set_xlabel('Nó destino')
ax.set_ylabel('Nó origem')
plt.tight_layout()
plt.savefig('../results/distance_matrix_heatmap.png', dpi=150)
plt.show()

print(f'Distância média entre pontos: {dist_array[dist_array > 0].mean():.1f} km')
print(f'Distância máxima: {dist_array.max():.1f} km')
print(f'Distância mínima (não-zero): {dist_array[dist_array > 0].min():.2f} km')

In [ ]:
# Resumo final
print('\n=== RESUMO DOS DADOS GERADOS ===')
print(f'Total de pontos: {len(points)}')
print(f'Demanda total: {df["demand"].sum()} unidades')
print(f'Capacidade do veículo: 120 unidades')
print(f'Rotas necessárias estimadas: {df["demand"].sum() / 120:.1f}')
print(f'\nDistância depósito → cada ponto (min/max):')
depot_dists = [dist_matrix[0][p.id] for p in points]
print(f'  Min: {min(depot_dists):.1f} km ({points[depot_dists.index(min(depot_dists))].name})')
print(f'  Max: {max(depot_dists):.1f} km ({points[depot_dists.index(max(depot_dists))].name})')